# DERMAINTEL — Multimodal MLP Training Pipeline

This notebook is a straight conversion of `03_train_multimodal_mlp.py` into runnable cells — **no code was changed**, it was only split into blocks (following the script's own section comments) so it's easier to run and inspect step by step.

Run the cells in order from top to bottom. The last cell calls `main()`, which runs the full pipeline (cross-validation, final model training, evaluation, permutation importance, smoothness experiment, plots, and README) exactly as the original script did.

### Imports & module docstring
_(lines 1-63 of the original script)_

In [1]:
"""
DERMAINTEL — 03_train_multimodal_mlp.py
==========================================

Trains and evaluates the multimodal fusion MLP that predicts a continuous
skin Risk Score from 256-D CNN image features + 5 environmental variables.

This script does NOT merge data and does NOT retrain the CNN. It reads the
already-merged merged_multimodal_dataset.csv (produced by
02_merge_multimodal_dataset.py) and trains only the small MLP head on top
of the already-extracted features.

Design decisions made beyond the literal spec (flagged explicitly, not
silently assumed):

1. SCALING. The 261 raw input columns (256 CNN features + 5 environmental
   variables) are on wildly different numeric scales (e.g. AQI_PM25 ~10-200
   vs UV_Index 0-11 vs ReLU-activation CNN features). Feeding these
   unscaled into a Dense+L2+Dropout network would make training unstable
   and would badly distort permutation-importance comparisons across
   variables of different scale. A StandardScaler is fit ONLY on training
   data (per-fold during cross-validation, and once on the full train split
   for the final model) and applied to validation/test data — never fit on
   anything the model doesn't train on, to avoid leakage. The fitted final
   scaler is saved (feature_scaler.pkl) since any future inference on new
   images needs the exact same transform.

2. SPLIT USAGE. The dataset already carries a Split column (train/val/test)
   inherited from the original CNN training pipeline. This script respects
   those boundaries rather than inventing a new split:
     - GroupKFold(5) cross-validation runs ONLY on Split == 'train' rows
       (grouped by Image_ID), so CV never touches val/test data.
     - The final model is trained on Split == 'train', using Split == 'val'
       as the validation_data for EarlyStopping/ReduceLROnPlateau/
       ModelCheckpoint.
     - "Final Model Evaluation" (MAE/RMSE/R²/Pearson) runs on Split ==
       'test' — data the final model never saw during training or CV.
"""

import os
import sys
import time
import pickle
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

### Config
_(lines 65-90 of the original script)_

In [2]:
# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

MERGED_CSV_PATH = r"C:\Users\GAURAV\Downloads\dermaintel-synth-delivery\outputs\merged_multimodal_dataset.csv"

FEATURE_COLS = [f"feature_{i:03d}" for i in range(256)]
ENV_COLS = ["Temperature", "Humidity", "UV_Index", "AQI_PM25", "Stress_Penalty"]
ALL_INPUT_COLS = FEATURE_COLS + ENV_COLS  # fixed order, used everywhere
TARGET_COL = "Risk_Score"

INPUT_DIM = len(ALL_INPUT_COLS)  # 261
LEARNING_RATE = 0.001
MAX_EPOCHS = 100
BATCH_SIZE = 32
N_CV_FOLDS = 5
RANDOM_SEED = 42

# Heuristic constants (must match the original synthetic-data generator
# exactly, since the smoothness experiment overlays this formula)
DISEASE_MULTIPLIERS = {"Acne": 1.1, "Eczema": 1.3, "Alopecia": 1.2, "Healthy": 0.8}
AQI_BREAKPOINTS = [(30, 0), (60, 1), (90, 2)]
AQI_ABOVE_TOP_SCORE = 3

OUTPUT_DIR = Path("outputs") / "mlp_training"

### Logging helper (Tee)
_(lines 92-109 of the original script)_

In [3]:
# ---------------------------------------------------------------------------
# LOGGING (tee stdout to both console and training_log.txt)
# ---------------------------------------------------------------------------

class Tee:
    """Duplicates writes to multiple streams — used to mirror all console
    output into training_log.txt without changing every print() call."""

    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)

    def flush(self):
        for s in self.streams:
            s.flush()

### Hardware configuration
_(lines 112-135 of the original script)_

In [4]:
# ---------------------------------------------------------------------------
# HARDWARE CONFIGURATION
# ---------------------------------------------------------------------------

def configure_hardware() -> bool:
    """Print GPU availability and enable mixed precision if a GPU is present."""
    print(f"TensorFlow version : {tf.__version__}")
    gpus = tf.config.list_physical_devices("GPU")
    print(f"Visible GPUs        : {gpus}")

    mixed_precision_enabled = False
    if gpus:
        print(f"GPU detected: {gpus[0].name}")
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            mixed_precision_enabled = True
            print("Mixed precision (mixed_float16) ENABLED.")
        except Exception as e:
            print(f"Could not enable mixed precision, continuing in float32: {e}")
    else:
        print("WARNING: No GPU detected. Training will use CPU (float32).")

    return mixed_precision_enabled

### Data loading
_(lines 138-166 of the original script)_

In [5]:
# ---------------------------------------------------------------------------
# DATA LOADING
# ---------------------------------------------------------------------------

def load_dataset(path: str) -> pd.DataFrame:
    """Load the already-merged multimodal dataset. Does not modify it."""
    if not Path(path).exists():
        print(f"ERROR: '{path}' not found. Run 02_merge_multimodal_dataset.py first.")
        sys.exit(1)

    df = pd.read_csv(path)

    missing = [c for c in ALL_INPUT_COLS + [TARGET_COL, "Image_ID", "Split", "Disease_Class"] if c not in df.columns]
    if missing:
        print(f"ERROR: merged dataset is missing expected columns: {missing}")
        sys.exit(1)

    return df


def print_dataset_summary(df: pd.DataFrame) -> None:
    print("\n--- Dataset Summary ---")
    print(f"Dataset size (rows)         : {len(df)}")
    print(f"Number of unique images     : {df['Image_ID'].nunique()}")
    profiles_per_image = df.groupby("Image_ID").size()
    print(f"Environmental profiles/image: {profiles_per_image.iloc[0]} "
          f"(min={profiles_per_image.min()}, max={profiles_per_image.max()})")
    print("Rows per split:")
    print(df["Split"].value_counts().to_string())

### Model definition & callbacks
_(lines 169-213 of the original script)_

In [6]:
# ---------------------------------------------------------------------------
# MODEL DEFINITION (fixed architecture — do not add layers/complexity)
# ---------------------------------------------------------------------------

def build_model(input_dim: int = INPUT_DIM, learning_rate: float = LEARNING_RATE) -> keras.Model:
    """
    Fixed architecture:
      Input(261) -> Dense(128, relu, L2) -> Dropout(0.5)
                  -> Dense(64, relu, L2)  -> Dropout(0.5)
                  -> Dense(1, linear)
    The output layer is forced to float32 regardless of the global mixed
    precision policy — standard practice to avoid numerical instability in
    the final regression output.
    """
    inputs = keras.Input(shape=(input_dim,), name="multimodal_input")
    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4),
                      name="dense_128")(inputs)
    x = layers.Dropout(0.5, name="dropout_1")(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4),
                      name="dense_64")(x)
    x = layers.Dropout(0.5, name="dropout_2")(x)
    outputs = layers.Dense(1, activation="linear", dtype="float32", name="risk_score_output")(x)

    model = keras.Model(inputs, outputs, name="multimodal_risk_mlp")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=["mae", keras.metrics.RootMeanSquaredError(name="rmse")],
    )
    return model


def make_callbacks(checkpoint_path: str = None, patience: int = 10) -> list:
    """Build the fixed callback set. checkpoint_path=None skips ModelCheckpoint
    (used during throwaway cross-validation folds, where we don't need to
    persist every fold's model to disk)."""
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=0),
    ]
    if checkpoint_path is not None:
        callbacks.append(
            ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, verbose=0)
        )
    return callbacks

### Cross-validation (GroupKFold)
_(lines 216-289 of the original script)_

In [7]:
# ---------------------------------------------------------------------------
# CROSS-VALIDATION (GroupKFold on Image_ID, within Split == 'train' only)
# ---------------------------------------------------------------------------

def run_cross_validation(train_df: pd.DataFrame, n_splits: int = N_CV_FOLDS) -> pd.DataFrame:
    """
    GroupKFold cross-validation, grouped by Image_ID, run only on the
    training split. Every one of an image's 8 environmental profiles stays
    together in the same fold, preventing leakage between train/validation
    within CV.
    """
    X = train_df[ALL_INPUT_COLS].values
    y = train_df[TARGET_COL].values
    groups = train_df["Image_ID"].values

    gkf = GroupKFold(n_splits=n_splits)
    fold_rows = []

    for fold_num, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        scaler = StandardScaler().fit(X_tr)
        X_tr_scaled = scaler.transform(X_tr)
        X_val_scaled = scaler.transform(X_val)

        model = build_model()
        model.fit(
            X_tr_scaled, y_tr,
            validation_data=(X_val_scaled, y_val),
            epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
            callbacks=make_callbacks(checkpoint_path=None),
            verbose=0,
        )

        preds = model.predict(X_val_scaled, verbose=0).flatten()
        mae = mean_absolute_error(y_val, preds)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        r2 = r2_score(y_val, preds)

        print(f"  Fold {fold_num}/{n_splits} — MAE: {mae:.4f}  RMSE: {rmse:.4f}  R2: {r2:.4f}  "
              f"(train rows={len(tr_idx)}, val rows={len(val_idx)})")

        fold_rows.append({"fold": fold_num, "mae": mae, "rmse": rmse, "r2": r2})

        keras.backend.clear_session()  # free memory between folds

    return pd.DataFrame(fold_rows)


def compute_ci(values: np.ndarray, confidence: float = 0.95) -> tuple:
    """Mean, std, and a t-distribution-based confidence interval — correct
    for small sample sizes (5 CV folds), unlike a naive z-based CI."""
    n = len(values)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if n > 1 else 0.0
    if n > 1:
        sem = std / np.sqrt(n)
        ci_low, ci_high = stats.t.interval(confidence, df=n - 1, loc=mean, scale=sem)
    else:
        ci_low, ci_high = mean, mean
    return mean, std, (float(ci_low), float(ci_high))


def summarize_cross_validation(fold_metrics: pd.DataFrame) -> pd.DataFrame:
    """Builds the mean/std/95% CI table for MAE, RMSE, R² across folds."""
    rows = []
    for metric in ["mae", "rmse", "r2"]:
        mean, std, (ci_low, ci_high) = compute_ci(fold_metrics[metric].values)
        rows.append({
            "metric": metric, "mean": mean, "std": std,
            "ci_95_low": ci_low, "ci_95_high": ci_high,
        })
    return pd.DataFrame(rows)

### Final model: train + evaluate
_(lines 292-390 of the original script)_

In [8]:
# ---------------------------------------------------------------------------
# FINAL MODEL: TRAIN + EVALUATE
# ---------------------------------------------------------------------------

def train_final_model(df: pd.DataFrame, output_dir: Path):
    """Trains the final model on Split=='train', validating on Split=='val'."""
    train_df = df[df["Split"] == "train"].reset_index(drop=True)
    val_df = df[df["Split"] == "val"].reset_index(drop=True)

    scaler = StandardScaler().fit(train_df[ALL_INPUT_COLS].values)
    X_train = scaler.transform(train_df[ALL_INPUT_COLS].values)
    y_train = train_df[TARGET_COL].values
    X_val = scaler.transform(val_df[ALL_INPUT_COLS].values)
    y_val = val_df[TARGET_COL].values

    model = build_model()
    n_params = model.count_params()
    print(f"Number of trainable parameters: {n_params}")

    best_model_path = str(output_dir / "best_model.keras")
    callbacks = make_callbacks(checkpoint_path=best_model_path, patience=10)

    print("\nTraining final model...")
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1,
    )
    training_time = time.time() - start_time
    print(f"Training time: {training_time:.1f} seconds")

    return model, scaler, history, training_time, n_params


def evaluate_final_model(model: keras.Model, scaler: StandardScaler, df: pd.DataFrame) -> dict:
    """Evaluates the final model on Split=='test' — fully held-out data."""
    test_df = df[df["Split"] == "test"].reset_index(drop=True)
    X_test = scaler.transform(test_df[ALL_INPUT_COLS].values)
    y_test = test_df[TARGET_COL].values

    preds = model.predict(X_test, verbose=0).flatten()

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    pearson_r, pearson_p = stats.pearsonr(y_test, preds)

    predictions_df = test_df[["Image_ID", "Profile_ID", "Disease_Class"]].copy()
    predictions_df["Actual_Risk_Score"] = y_test
    predictions_df["Predicted_Risk_Score"] = preds
    predictions_df["Residual"] = y_test - preds

    return {
        "mae": mae, "rmse": rmse, "r2": r2,
        "pearson_r": pearson_r, "pearson_p": pearson_p,
        "y_test": y_test, "preds": preds,
        "predictions_df": predictions_df,
    }


def interpret_fit_quality(history: keras.callbacks.History, test_metrics: dict) -> str:
    """
    A simple, transparent, rule-based read on whether the model looks like
    it's underfitting, overfitting, or learning a reasonable smooth fit —
    based on final train/val loss and test R², not a black-box judgement.
    """
    final_train_loss = history.history["loss"][-1]
    final_val_loss = history.history["val_loss"][-1]
    r2 = test_metrics["r2"]

    lines = [
        f"Final training loss (MSE)  : {final_train_loss:.4f}",
        f"Final validation loss (MSE): {final_val_loss:.4f}",
        f"Test R^2                   : {r2:.4f}",
        "",
    ]

    if r2 < 0.3 and final_train_loss > 1.0:
        verdict = ("The model appears to be UNDERFITTING: both training loss remains high "
                   "and test R^2 is low. The network is not capturing enough of the "
                   "relationship between inputs and Risk_Score.")
    elif final_val_loss > 1.5 * final_train_loss and r2 < 0.6:
        verdict = ("The model shows signs of OVERFITTING: validation loss is substantially "
                   "higher than training loss, and test R^2 is only moderate. Consider more "
                   "regularization, more data, or fewer epochs.")
    elif r2 >= 0.6:
        verdict = ("The model appears to be learning a SMOOTH, reasonable approximation of "
                   "the heuristic Risk Score: training and validation loss are reasonably "
                   "close, and test R^2 indicates the model explains a meaningful share of "
                   "the variance.")
    else:
        verdict = ("Results are mixed — neither clearly underfitting nor overfitting. "
                   "Inspect the loss curve and actual-vs-predicted plot directly before "
                   "drawing a firm conclusion.")

    lines.append(verdict)
    return "\n".join(lines)

### Permutation importance
_(lines 393-429 of the original script)_

In [9]:
# ---------------------------------------------------------------------------
# PERMUTATION IMPORTANCE (environmental variables only — no SHAP)
# ---------------------------------------------------------------------------

def compute_permutation_importance(
    model: keras.Model, scaler: StandardScaler, test_df: pd.DataFrame,
    n_repeats: int = 10, seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """
    Manual permutation importance, restricted to the 5 environmental
    variables (never the 256 CNN features). For each variable: shuffle it
    across test rows, re-predict, measure the INCREASE in MAE vs baseline.
    Larger increase = more important. Explicitly not SHAP, per spec.
    """
    rng = np.random.default_rng(seed)
    X_raw = test_df[ALL_INPUT_COLS].values.copy()
    y_test = test_df[TARGET_COL].values

    baseline_preds = model.predict(scaler.transform(X_raw), verbose=0).flatten()
    baseline_mae = mean_absolute_error(y_test, baseline_preds)

    rows = []
    for col in ENV_COLS:
        col_idx = ALL_INPUT_COLS.index(col)
        repeat_maes = []
        for _ in range(n_repeats):
            X_permuted = X_raw.copy()
            X_permuted[:, col_idx] = rng.permutation(X_permuted[:, col_idx])
            preds = model.predict(scaler.transform(X_permuted), verbose=0).flatten()
            repeat_maes.append(mean_absolute_error(y_test, preds))
        importance = float(np.mean(repeat_maes) - baseline_mae)
        rows.append({"variable": col, "importance_mae_increase": importance,
                      "std_across_repeats": float(np.std(repeat_maes))})

    importance_df = pd.DataFrame(rows).sort_values("importance_mae_increase", ascending=False).reset_index(drop=True)
    importance_df.attrs["baseline_mae"] = baseline_mae
    return importance_df

### Smoothness experiment
_(lines 432-489 of the original script)_

In [10]:
# ---------------------------------------------------------------------------
# SMOOTHNESS EXPERIMENT
# ---------------------------------------------------------------------------

def aqi_category_score(pm25: float) -> int:
    """Same AQI -> Environmental Score mapping used by the synthetic generator."""
    for upper_bound, score in AQI_BREAKPOINTS:
        if pm25 <= upper_bound:
            return score
    return AQI_ABOVE_TOP_SCORE


def run_smoothness_experiment(
    df: pd.DataFrame, model: keras.Model, scaler: StandardScaler, seed: int = RANDOM_SEED,
) -> dict:
    """
    Picks one random NON-Healthy image (Healthy's Risk_Score is always 0 by
    design, which would make the heuristic overlay a flat, uninformative
    line), holds its 256 CNN features and all environmental variables fixed
    except UV_Index, sweeps UV_Index continuously from 0 to 11, and compares
    the MLP's smooth prediction curve against the original heuristic
    formula's curve (a genuine step function) at the same fixed values.
    """
    rng = np.random.default_rng(seed)
    non_healthy = df[df["Disease_Class"] != "Healthy"]
    chosen_image_id = rng.choice(non_healthy["Image_ID"].unique())
    row = non_healthy[non_healthy["Image_ID"] == chosen_image_id].iloc[0]

    base_features = row[FEATURE_COLS].values.astype(float)
    base_env = row[ENV_COLS].values.astype(float).copy()
    uv_col_idx = ENV_COLS.index("UV_Index")

    uv_grid = np.linspace(0, 11, 200)
    grid_rows = []
    for uv in uv_grid:
        env = base_env.copy()
        env[uv_col_idx] = uv
        grid_rows.append(np.concatenate([base_features, env]))
    X_grid = np.array(grid_rows)
    mlp_preds = model.predict(scaler.transform(X_grid), verbose=0).flatten()

    temperature, humidity, _, aqi_pm25, stress_penalty = base_env
    disease_multiplier = DISEASE_MULTIPLIERS[row["Disease_Class"]]
    heuristic_preds = []
    for uv in uv_grid:
        es = aqi_category_score(aqi_pm25)
        es += 2 if uv >= 8 else (1 if uv >= 5 else 0)
        es += 1 if (humidity >= 80 or humidity <= 20) else 0
        ls = stress_penalty
        heuristic_preds.append(disease_multiplier * (ls + es))

    return {
        "image_id": chosen_image_id,
        "disease_class": row["Disease_Class"],
        "uv_grid": uv_grid,
        "mlp_preds": mlp_preds,
        "heuristic_preds": np.array(heuristic_preds),
    }

### Plots
_(lines 492-561 of the original script)_

In [11]:
# ---------------------------------------------------------------------------
# PLOTS
# ---------------------------------------------------------------------------

def plot_loss_curve(history: keras.callbacks.History, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(history.history["loss"], label="Training Loss")
    ax.plot(history.history["val_loss"], label="Validation Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.set_title("Training vs Validation Loss")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "loss_curve.png", dpi=120)
    plt.close(fig)


def plot_actual_vs_predicted(y_true: np.ndarray, y_pred: np.ndarray, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, alpha=0.5, s=18, color="#3B6FE0")
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", label="Perfect prediction")
    ax.set_xlabel("Actual Risk Score")
    ax.set_ylabel("Predicted Risk Score")
    ax.set_title("Actual vs Predicted Risk Score (Test Set)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "actual_vs_predicted.png", dpi=120)
    plt.close(fig)


def plot_residuals(y_true: np.ndarray, y_pred: np.ndarray, output_dir: Path) -> None:
    residuals = y_true - y_pred
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(y_pred, residuals, alpha=0.5, s=18, color="#22D3C7")
    ax.axhline(0, color="gray", linestyle="--")
    ax.set_xlabel("Predicted Risk Score")
    ax.set_ylabel("Residual (Actual - Predicted)")
    ax.set_title("Residual Plot (Test Set)")
    fig.tight_layout()
    fig.savefig(output_dir / "residual_plot.png", dpi=120)
    plt.close(fig)


def plot_feature_importance(importance_df: pd.DataFrame, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(importance_df["variable"], importance_df["importance_mae_increase"], color="#3B6FE0")
    ax.set_xlabel("Increase in MAE when permuted (higher = more important)")
    ax.set_title("Environmental Feature Importance (Permutation)")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(output_dir / "environment_feature_importance.png", dpi=120)
    plt.close(fig)


def plot_uv_smoothness(smoothness_result: dict, output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.plot(smoothness_result["uv_grid"], smoothness_result["mlp_preds"],
            label="MLP prediction", color="#3B6FE0", linewidth=2)
    ax.plot(smoothness_result["uv_grid"], smoothness_result["heuristic_preds"],
            label="Heuristic formula", color="#E0673B", linewidth=2, linestyle="--")
    ax.set_xlabel("UV Index")
    ax.set_ylabel("Predicted Risk Score")
    ax.set_title(f"MLP vs Heuristic — UV Index sweep\n"
                 f"(Image: {smoothness_result['image_id']}, Class: {smoothness_result['disease_class']})")
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "uv_smoothness_curve.png", dpi=120)
    plt.close(fig)

### README writer
_(lines 563-630 of the original script)_

In [12]:
# ---------------------------------------------------------------------------
# README
# ---------------------------------------------------------------------------

def write_readme(output_dir: Path) -> None:
    readme_text = """DERMAINTEL — Multimodal MLP Training Outputs
================================================

Models
------
mlp_model.keras   : final model's in-memory weights at the end of training
                    (EarlyStopping restore_best_weights=True, so this is
                    already the best-val-loss epoch's weights).
best_model.keras  : the best model checkpoint saved during training via
                    ModelCheckpoint (monitor='val_loss', save_best_only).
                    Should closely match mlp_model.keras.
feature_scaler.pkl: the StandardScaler fit on the final model's training
                    split. Required for any future inference — new inputs
                    MUST be transformed with this exact scaler before being
                    passed to the model.

Cross-validation (GroupKFold, 5 folds, grouped by Image_ID, Split=='train' only)
---------------------------------------------------------------------------------
fold_metrics.csv            : one row per fold — MAE, RMSE, R2 on that
                               fold's held-out validation group.
cross_validation_results.csv: aggregated MAE/RMSE/R2 across all folds —
                               mean, std, and 95% confidence interval.
cross_validation_summary.txt: the same aggregated CV results, as readable text.

Final model training + evaluation
----------------------------------
training_history.csv : per-epoch loss/mae/rmse for the FINAL model's
                        training run (train and validation).
training_log.txt      : full console output of the entire run.
model_summary.txt     : keras model.summary() text for the final model.
evaluation_metrics.txt: final model's MAE/RMSE/R2/Pearson correlation on
                        the held-out TEST split, plus a plain-language
                        under/over/well-fit interpretation.
predictions.csv        : per-row test-set predictions (Image_ID, Profile_ID,
                        Disease_Class, Actual_Risk_Score,
                        Predicted_Risk_Score, Residual).

Environmental feature importance (permutation, test split, NOT SHAP)
------------------------------------------------------------------------
permutation_importance.csv: MAE increase when each of the 5 environmental
                            variables is independently shuffled — larger
                            increase = more important to the model's
                            predictions. CNN features are never permuted.

Smoothness experiment
----------------------
uv_smoothness_curve.png shows, for one randomly chosen NON-Healthy image
(Healthy's Risk_Score is always 0 by design and wouldn't show anything),
the MLP's prediction as UV_Index is swept continuously from 0 to 11 with
every other input held fixed, overlaid against what the original heuristic
formula would output at those same UV values. The heuristic is a genuine
step function (jumps at UV=5 and UV=8); the MLP curve's smoothness (or lack
of it) is the point of this experiment.

Plots
-----
loss_curve.png                    : final model's training vs validation loss per epoch.
actual_vs_predicted.png           : scatter of actual vs predicted Risk Score (test split).
residual_plot.png                 : predicted value vs residual (test split).
environment_feature_importance.png: bar chart of the 5 environmental variables' permutation importance.
uv_smoothness_curve.png           : MLP vs heuristic curve as UV_Index varies.
"""
    (output_dir / "README.txt").write_text(readme_text, encoding="utf-8")

### main() function
_(lines 633-750 of the original script)_

In [13]:
# ---------------------------------------------------------------------------
# ENTRY POINT
# ---------------------------------------------------------------------------

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    log_file = open(OUTPUT_DIR / "training_log.txt", "w", encoding="utf-8")
    sys.stdout = Tee(sys.__stdout__, log_file)

    try:
        tf.random.set_seed(RANDOM_SEED)
        np.random.seed(RANDOM_SEED)

        print("=" * 70)
        print("DERMAINTEL — Multimodal MLP Training Pipeline")
        print("=" * 70)

        configure_hardware()

        df = load_dataset(MERGED_CSV_PATH)
        print_dataset_summary(df)

        # ---------------- Cross-validation ----------------
        print("\n" + "=" * 70)
        print(f"CROSS-VALIDATION: GroupKFold({N_CV_FOLDS} folds), grouped by Image_ID, Split=='train' only")
        print("=" * 70)
        train_only_df = df[df["Split"] == "train"].reset_index(drop=True)
        fold_metrics = run_cross_validation(train_only_df, n_splits=N_CV_FOLDS)
        fold_metrics.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)

        cv_summary = summarize_cross_validation(fold_metrics)
        cv_summary.to_csv(OUTPUT_DIR / "cross_validation_results.csv", index=False)

        print("\nCross-validation summary (mean / std / 95% CI):")
        print(cv_summary.to_string(index=False))
        cv_summary_text = "DERMAINTEL — Cross-Validation Summary\n" + "=" * 40 + "\n\n" + cv_summary.to_string(index=False)
        (OUTPUT_DIR / "cross_validation_summary.txt").write_text(cv_summary_text, encoding="utf-8")

        # ---------------- Final model ----------------
        print("\n" + "=" * 70)
        print("FINAL MODEL: training on Split=='train', validating on Split=='val'")
        print("=" * 70)
        model, scaler, history, training_time, n_params = train_final_model(df, OUTPUT_DIR)

        model.save(OUTPUT_DIR / "mlp_model.keras")
        with open(OUTPUT_DIR / "feature_scaler.pkl", "wb") as f:
            pickle.dump(scaler, f)

        pd.DataFrame(history.history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

        summary_lines = []
        model.summary(print_fn=lambda x: summary_lines.append(x))
        (OUTPUT_DIR / "model_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")

        plot_loss_curve(history, OUTPUT_DIR)

        # ---------------- Final evaluation (held-out test split) ----------------
        print("\n" + "=" * 70)
        print("FINAL MODEL EVALUATION (Split=='test', fully held out)")
        print("=" * 70)
        test_results = evaluate_final_model(model, scaler, df)
        print(f"MAE     : {test_results['mae']:.4f}")
        print(f"RMSE    : {test_results['rmse']:.4f}")
        print(f"R2      : {test_results['r2']:.4f}")
        print(f"Pearson correlation (Predicted vs Heuristic Risk): "
              f"r={test_results['pearson_r']:.4f}  (p={test_results['pearson_p']:.4g})")

        interpretation = interpret_fit_quality(history, test_results)
        print("\nFit interpretation:")
        print(interpretation)

        evaluation_text = (
            "DERMAINTEL — Final Model Evaluation (Split == 'test')\n" + "=" * 55 + "\n\n"
            f"MAE  : {test_results['mae']:.4f}\n"
            f"RMSE : {test_results['rmse']:.4f}\n"
            f"R2   : {test_results['r2']:.4f}\n"
            f"Pearson correlation (Predicted vs Heuristic Risk): "
            f"r={test_results['pearson_r']:.4f} (p={test_results['pearson_p']:.4g})\n\n"
            f"{interpretation}\n"
        )
        (OUTPUT_DIR / "evaluation_metrics.txt").write_text(evaluation_text, encoding="utf-8")

        test_results["predictions_df"].to_csv(OUTPUT_DIR / "predictions.csv", index=False)
        plot_actual_vs_predicted(test_results["y_test"], test_results["preds"], OUTPUT_DIR)
        plot_residuals(test_results["y_test"], test_results["preds"], OUTPUT_DIR)

        # ---------------- Permutation importance ----------------
        print("\n" + "=" * 70)
        print("ENVIRONMENTAL FEATURE IMPORTANCE (Permutation, test split)")
        print("=" * 70)
        test_df = df[df["Split"] == "test"].reset_index(drop=True)
        importance_df = compute_permutation_importance(model, scaler, test_df)
        print(importance_df.to_string(index=False))
        importance_df.to_csv(OUTPUT_DIR / "permutation_importance.csv", index=False)
        plot_feature_importance(importance_df, OUTPUT_DIR)

        # ---------------- Smoothness experiment ----------------
        print("\n" + "=" * 70)
        print("SMOOTHNESS EXPERIMENT (UV_Index sweep, one random non-Healthy image)")
        print("=" * 70)
        smoothness_result = run_smoothness_experiment(df, model, scaler)
        print(f"Chosen image: {smoothness_result['image_id']} "
              f"(class: {smoothness_result['disease_class']})")
        plot_uv_smoothness(smoothness_result, OUTPUT_DIR)

        # ---------------- README ----------------
        write_readme(OUTPUT_DIR)

        print("\n" + "=" * 70)
        print("PIPELINE COMPLETE")
        print("=" * 70)
        print(f"Number of trainable parameters : {n_params}")
        print(f"Training time                  : {training_time:.1f} seconds")
        print(f"Output folder                  : {OUTPUT_DIR.resolve()}")

    finally:
        sys.stdout = sys.__stdout__
        log_file.close()

### Entry point
_(lines 753-754 of the original script)_

In [14]:
if __name__ == "__main__":
    main()